# Task 3a: YCSEP preparation and Parakeet fine-tuning

Status: CPU preparation implemented; GPU training and training-curve interpretation pending. This notebook is not a completed Task 3a deliverable.

Hypothesis: reliable, diverse Singapore conversational speech can improve domain adaptation per GPU-hour. Improvement is a hypothesis, not an assumed result.

In [ ]:
import json, os, sys
from pathlib import Path
root = Path.cwd().resolve()
if root.name == 'asr-train':
    root = root.parent
sys.path.insert(0, str(root / 'asr-train'))
from prepare_metadata import prepare
source = Path(os.environ.get('YCSEP_CSV', str(root / 'test_docs/test/runtime/local-prep-0316/data/YCSEP_static.csv')))
output = Path(os.environ.get('YCSEP_AUDIT', str(root / 'test_docs/test/runtime/metadata-audit')))
print('Source:', source)
print('Output:', output)

## Freeze the split before curation

All TDK source-video IDs and audio URLs are held out. Non-TDK videos receive a deterministic hash-based train/validation split, seed 2026, expected validation fraction 10%. The realized fraction need not equal 10%. Video grouping reduces adjacent-clip leakage but does not prove speaker independence. Near-duplicate audio requires a later acoustic check.

The CPU candidate policy retains original transcripts and Singlish. Invalid timestamps, empty lexical text, invalid URLs, duplicate URLs and durations outside 0.3-20 seconds are excluded from the pilot candidate pool. Sparse/dense text, fillers and repeated characters remain flags. Validation receives only objective validity checks; later learned quality thresholds must not redefine it to improve reported performance.

In [ ]:
assert source.exists(), 'Set YCSEP_CSV to the supplied YCSEP_static.csv'
audit = prepare(source, output)
audit

## Inspect duration, text and overlap

Segment-hours may count overlapping intervals more than once. The sweep-line overlap report separately measures union time and time covered by at least two annotation intervals. Annotation overlap is not proof of simultaneous speech. TDK metadata is reported for coverage, not used to tune quality rules.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(list(audit['channel_segment_hours']), list(audit['channel_segment_hours'].values()))
axes[0].set_xlabel('Annotated segment-hours (not unique audio hours)')
axes[1].bar(list(audit['duration_buckets']), list(audit['duration_buckets'].values()))
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Segments')
fig.tight_layout()
fig.savefig(output / 'metadata_overview.png', dpi=160)
plt.show()

## Audio preparation and feature extraction

Next: sample a diverse ~10-hour pilot from training candidates, freeze a non-TDK validation sample, download on CPU, validate nonempty finite waveforms and actual duration, and convert to mono 16 kHz WAV. Candidate JSONL files are not yet NeMo training manifests. Audio-ready manifests require audio_filepath, measured duration and original text, plus source-video provenance.

Retain Parakeet's pretrained SentencePiece tokenizer and acoustic preprocessor. Log the loaded tokenizer vocabulary and preprocessor configuration rather than guessing feature parameters. Inspect Singlish tokenization without retraining the tokenizer. Use VAD only on suspicious sampled cases initially; defer MOS models unless they change a demonstrated data decision.

## Planned PyTorch/NeMo experiment

E0: base model WER/CER on frozen non-TDK validation. E1: full fine-tuning on a ~10-hour pilot, initially AdamW learning rate 1e-5, weight decay 0.001, batch size 8, gradient accumulation 4, BF16 on supported GPU, gradient clipping 1.0. These are starting values, not optimized settings. Profile memory and steps/second before setting the final duration budget. Reuse model-compatible SpecAugment; do not add unverified augmentation defaults.

Validate early enough to produce a usable checkpoint even if the time budget expires. Keep restart state and select lowest validation WER. Record loss, validation WER/CER, steps, actual hours, optimizer, seed, package versions and elapsed training time. Scale to 25-50 hours only if the pilot is healthy; no TDK-based model selection.

Final artifact: parakeet-tdt-0.6b-v3-ycsep.nemo. Training implementation, executed loss/WER plots and evidence-based interpretation remain to be completed.

## Interpretation to complete after execution

Compare E0 and E1 on the same validation rows. Decreasing training loss alone does not demonstrate generalization. Inspect worsening validation WER, plateau, corrupted batches and subgroup regressions before scaling. Report failed audio coverage separately. Never replace a missing experiment with a fabricated curve.

References: [Parakeet model card](https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3), [JiWER](https://jitsi.github.io/jiwer/).